# Swiss Job Market Intelligence — Exploratory Data Analysis

**Pipeline:** Python (ingestion) → dbt (transformation) → PostgreSQL (warehouse) → this notebook

This notebook explores the `analytics` schema marts built by dbt:
- `mart_market_overview` — postings by city/industry/month with salary & remote-work stats
- `mart_skill_demand` — skill frequency and pay premium by city
- `mart_salary_benchmarks` — salary distributions by role, seniority, and city

Author: Victoria Esther


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine

# Brand palette — purple, matching the Tableau dashboard
PURPLE = "#7C3AED"
PURPLE_PALETTE = ["#F3E8FF", "#D8B4FE", "#C084FC", "#A855F7", "#9333EA", "#7C3AED", "#6D28D9", "#5B21B6"]

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#E5E7EB"
plt.rcParams["axes.titleweight"] = "bold"

%matplotlib inline


## Connect to the data warehouse

Connection details point at the local PostgreSQL instance populated by the dbt project (`swiss_jobs` database, `analytics` schema).

In [ ]:
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "swiss_jobs"
DB_USER = "postgres"
DB_PASSWORD = "SwissJobs2026!"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

df_market = pd.read_sql("SELECT * FROM analytics.mart_market_overview", engine)
df_skills = pd.read_sql("SELECT * FROM analytics.mart_skill_demand", engine)
df_salary = pd.read_sql("SELECT * FROM analytics.mart_salary_benchmarks", engine)

df_market["month_year"] = pd.to_datetime(df_market["month_year"])

print(f"market_overview: {df_market.shape}, skill_demand: {df_skills.shape}, salary_benchmarks: {df_salary.shape}")


## 1. Job postings by Swiss city

**Business insight:** Demand for data & tech roles is heavily concentrated in Zurich, which should anchor any hiring, relocation, or salary-benchmarking strategy. Secondary hubs (Geneva, Basel, Lausanne) make up a meaningful but much smaller share — useful for candidates open to lower-competition markets or employers looking to diversify talent pools outside the capital-intensive Zurich market.

In [ ]:
city_postings = (
    df_market.groupby("city_clean")["postings_count"].sum().sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(city_postings.index, city_postings.values, color=PURPLE, edgecolor="white")
ax.bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9)
ax.set_title("Job Postings by Swiss City — 2025–2026")
ax.set_xlabel("City")
ax.set_ylabel("Postings Count")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.xticks(rotation=30, ha="right")
sns.despine()
plt.tight_layout()
plt.show()


## 2. Top 15 in-demand skills

**Business insight:** Python and SQL dominate postings by a wide margin, confirming they're table-stakes for nearly every data role in this market. The long tail (Airflow, Spark, Kafka, cloud platforms) signals where specialization commands differentiation — useful for prioritizing upskilling or curriculum design.

In [ ]:
top_skills = df_skills.groupby("skill")["job_count"].sum().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette("Purples", n_colors=len(top_skills))[::-1]
bars = ax.barh(top_skills.index[::-1], top_skills.values[::-1], color=colors[::-1])
ax.bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9)
ax.set_title("Top 15 Skills by Job Count")
ax.set_xlabel("Job Count")
ax.set_ylabel("Skill")
sns.despine()
plt.tight_layout()
plt.show()


## 3. Average salary by role

**Business insight:** Senior data science and engineering roles command a clear premium (150K+ CHF), while BI Developer and Data Analyst roles sit meaningfully lower. This ladder is useful for setting compensation bands, planning promotion paths, or evaluating whether a specific offer is competitive for the Swiss market.

In [ ]:
role_salary = df_salary.groupby("role")["avg_salary_chf"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette("Purples", n_colors=len(role_salary))[::-1]
bars = ax.barh(role_salary.index[::-1], role_salary.values[::-1], color=colors[::-1])
ax.bar_label(bars, fmt="CHF {:,.0f}", padding=3, fontsize=9)
ax.set_title("Average Salary by Role")
ax.set_xlabel("Avg. Salary (CHF)")
ax.set_ylabel("Role")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
sns.despine()
plt.tight_layout()
plt.show()


## 4. Remote-work share by city

**Business insight:** Remote-work prevalence varies notably by city — cities skewing higher likely have a heavier SaaS/tech employer mix, while lower-remote cities may be dominated by industries (banking, pharma, manufacturing) with more on-site culture. This helps candidates filter by lifestyle fit and helps employers benchmark their remote policy against the local norm.

In [ ]:
remote_by_city = df_market.groupby("city_clean")["remote_pct"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(remote_by_city.index, remote_by_city.values, color=PURPLE, edgecolor="white")
ax.bar_label(bars, fmt="{:.1f}%", padding=3, fontsize=9)
ax.set_title("Average Remote-Work % by City")
ax.set_xlabel("City")
ax.set_ylabel("Avg. Remote %")
plt.xticks(rotation=30, ha="right")
sns.despine()
plt.tight_layout()
plt.show()


## 5. Monthly posting trend

**Business insight:** Tracking postings month over month surfaces seasonality and momentum — a rising trend signals a tightening (candidate-favorable) market, while a plateau or decline suggests employers should expect more applicant competition per role. This view is the single best signal for timing a job search or a hiring push.

In [ ]:
monthly = df_market.groupby("month_year")["postings_count"].sum().sort_index()

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(monthly.index, monthly.values, marker="o", color=PURPLE, linewidth=2.5, markersize=6)
for x, y in zip(monthly.index, monthly.values):
    ax.annotate(f"{y:,.0f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_title("Monthly Job Posting Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Postings Count")
fig.autofmt_xdate(rotation=30)
sns.despine()
plt.tight_layout()
plt.show()


## 6. Salary heatmap — city × role

**Business insight:** The heatmap exposes where role-level pay is highest *within* a given city, not just on average nationally. A role might look well-paid in the bar chart above but actually be a top payer only in one or two hub cities — critical context for relocation decisions or city-specific compensation benchmarking.

In [ ]:
pivot = df_salary.pivot_table(index="city_clean", columns="role", values="avg_salary_chf", aggfunc="mean")

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot,
    annot=True,
    fmt=",.0f",
    cmap="Purples",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Avg. Salary (CHF)"},
    ax=ax,
)
ax.set_title("Average Salary (CHF) by City and Role")
ax.set_xlabel("Role")
ax.set_ylabel("City")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()


## Summary

- Zurich dominates posting volume; Geneva, Basel, and Lausanne are secondary hubs.
- Python and SQL are baseline requirements; specialization in cloud/streaming tools differentiates candidates.
- Senior Data Scientist and Senior Data Engineer roles top the pay ladder; BI Developer and Data Analyst sit lowest.
- Remote-work share varies by city, likely reflecting local industry mix.
- Posting volume shows clear month-to-month movement, useful for timing decisions.
- Pay by role is not uniform across cities — city-specific benchmarking beats a single national figure.

*Data: Swiss Job Market 2025–2026 | Pipeline: Python → dbt → PostgreSQL | Victoria Esther*